In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import statistics
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

# --- 0. CONSTANTS & GLOBAL SETUP ---
H5_FILE_PATH = "/home/poorna/data/eeg_visual_clip_vit_l14_final.h5" 
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

D_MODEL = 256  # Transformer hidden dimension
D_IMG = 768   # CLIP ViT-L/14 embedding dimension
NUM_CHANNELS = 62 
EPOCHS = 50
MODEL_SAVE_PATH = "/home/poorna/capstone/first_implementation/eeg_temporal_clip_embedding_transformer.pt"

# ====================================================================
# --- 1. DATASET & DATALOADER ---
# ====================================================================
class EEG2VisualH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        # Input: EEG (C, T)
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        # Target: Visual Embedding (D_IMG,)
        visual_embedding = torch.from_numpy(self.h5_file['visual_embeddings'][idx].astype('float32'))
        
        return eeg, visual_embedding

def collate_visual_batch(batch):
    eeg_list, embed_list = [], []
    for eeg, embed in batch:
        eeg_list.append(eeg)
        embed_list.append(embed)

    eeg_batch = torch.stack(eeg_list, dim=0) 
    embed_batch = torch.stack(embed_list, dim=0)

    return eeg_batch.float(), embed_batch.float()

# ====================================================================
# --- 2. MODEL ARCHITECTURE COMPONENTS (Temporal-only) ---
# ====================================================================

# --- Encoder Helpers ---
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None: mask = mask.unsqueeze(1)
        batch_size = query.size(0)
        
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=NUM_CHANNELS, d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        # 1. Spatial/Feature Projection (Conv1D)
        # REMOVE LayerNorm from here. It is applied after the transpose.
        self.channel_projector = nn.Sequential(
            nn.Conv1d(in_channels=num_channels, out_channels=d_model, kernel_size=1),
            nn.ReLU(),
        )

        # NEW: LayerNorm for the T dimensional input (applied after transpose)
        self.spatial_norm = nn.LayerNorm(d_model) 

        # 2. Temporal Processing (Transformer)
        self.pos_encoding = PositionalEncoding(d_model)
        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model) # This is the standard Transformer stack norm

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg):
        # eeg shape: (B, C, T)
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # 1. Spatial Projection (Conv1D)
        # Output x shape: (B, D_model, T) 
        x = self.channel_projector(eeg)
        
        # Transpose to get the sequence dimension first: (B, T, D_model)
        x = x.transpose(1, 2) # Shape: (32, 400, 256)
        
        # Apply LayerNorm to the feature dimension (256)
        x = self.spatial_norm(x)
        
        # 2. Temporal Processing (Transformer Encoder)
        cls_token = self.cls_token.repeat(batch_size, 1, 1) 
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        x = self.pos_encoding(x)

        for layer in self.transformer_layers:
            x = layer(x)
        
        return self.layer_norm(x)


class ImageProjectionHead(nn.Module):
    """Stage 1 Decoder: Projects the CLS token into the CLIP embedding space."""
    def __init__(self, d_model=D_MODEL, d_img=D_IMG):
        super().__init__()
        self.projection = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(d_model * 2, d_img)
        )
        
    def forward(self, cls_token_feature):
        return self.projection(cls_token_feature)

class EEG2ImageEmbed(nn.Module):
    def __init__(self, d_model=D_MODEL, d_img=D_IMG, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        
        self.encoder = TemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.image_head = ImageProjectionHead(d_model, d_img)

    def forward(self, eeg):
        # 1. Encode EEG (Temporal encoding only)
        eeg_features = self.encoder(eeg) # (B, T+1, D_model)
        
        # 2. Extract CLS Token
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        
        # 3. Predict Image Embedding
        image_embedding_pred = self.image_head(cls_token_feature) # (B, D_img)

        return image_embedding_pred

# ====================================================================
# --- 3. TRAINING AND EVALUATION FUNCTIONS (MSE Loss) ---
# ====================================================================

def train_one_epoch_visual(model, loader, optimizer, img_criterion):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training Visual Embed", leave=False)
    
    for eeg_b, embed_b in progress_bar:
        eeg_b, embed_b = eeg_b.to(device), embed_b.to(device)
        optimizer.zero_grad()
        
        embed_pred = model(eeg_b) # No graph inputs
        
        loss = img_criterion(embed_pred, embed_b) # MSE Loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(MSE=loss.item())
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_visual(model, loader, img_criterion):
    model.eval()
    total_loss = 0.0
    
    for eeg_b, embed_b in loader:
        eeg_b, embed_b = eeg_b.to(device), embed_b.to(device)
        embed_pred = model(eeg_b) # No graph inputs
        loss = img_criterion(embed_pred, embed_b)
        total_loss += loss.item()
        
    return total_loss / len(loader)

@torch.no_grad()
def calculate_cosine_similarity(model, loader):
    model.eval()
    all_cos_sims = []
    
    for eeg_b, embed_b in tqdm(loader, desc="Calculating Cosine Similarity", leave=False):
        eeg_b, embed_b = eeg_b.to(device), embed_b.to(device)
        embed_pred = model(eeg_b) # No graph inputs
        
        # Normalize predicted embedding before comparing (CLIP targets are already normalized)
        embed_pred_norm = F.normalize(embed_pred, p=2, dim=1)
        
        # Cosine Similarity is the key metric for embedding alignment
        cos_sim = F.cosine_similarity(embed_pred_norm, embed_b, dim=1)
        all_cos_sims.extend(cos_sim.cpu().tolist())
        
    return statistics.mean(all_cos_sims)



Using device: cuda


In [2]:
# ====================================================================
# --- 4. EXECUTION ---
# ====================================================================
if __name__ == '__main__':
    print("--- 1. Setting up Data ---")
    
    g = torch.Generator().manual_seed(42)
    dataset = EEG2VisualH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val   = int(N * VAL_PCT)
    n_test  = N - n_train - n_val
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_visual_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_visual_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_visual_batch)
    print(f"Data loaders created: Train={n_train}, Val={n_val}, Test={n_test}")
    
    # 2. Model & Loss Initialization
    model = EEG2ImageEmbed(d_model=D_MODEL, d_img=D_IMG).to(device)
    img_criterion = nn.MSELoss() 

    # 3. Training Loop Setup
    new_optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    # The scheduler now relies only on the required arguments
    new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=5)
    best_val_loss = float('inf')

    print(f"\n--- 2. Starting Temporal Embedding Training ---")
    print(f"Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        # No graph inputs passed to training functions
        train_loss = train_one_epoch_visual(model, train_loader, new_optimizer, img_criterion)
        val_loss = evaluate_visual(model, val_loader, img_criterion)
        new_scheduler.step(val_loss)
        end_time = time.time()
        formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
        
        print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
        print(f"\tTrain MSE Loss: {train_loss:.6f}")
        print(f"\t Val. MSE Loss: {val_loss:.6f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("\t-> Validation loss improved, saving new best model. 🏆")

    print("\n--- 3. Training Complete. Starting Final Evaluation ---")
    
    # 4. Final Evaluation
    try:
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
        
        # No graph inputs passed to evaluation functions
        test_mse = evaluate_visual(model, test_loader, img_criterion)
        test_cos_sim = calculate_cosine_similarity(model, test_loader)

        print("\n=============================================")
        print(f"| FINAL RESULTS on Test Set |")
        print("---------------------------------------------")
        print(f"| **Test MSE Loss:**         {test_mse:.6f} |")
        print(f"| **Test Cosine Similarity:** {test_cos_sim:.4f} |")
        print("=============================================")
    except Exception as e:
        print(f"Error during final evaluation: {e}. Ensure training completed successfully and '{MODEL_SAVE_PATH}' exists.")

--- 1. Setting up Data ---
Data loaders created: Train=22400, Val=2800, Test=2800

--- 2. Starting Temporal Embedding Training ---
Model Parameters: 3,702,528


Training Visual Embed:   0%|          | 0/700 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
print(device)

cuda
